# Conversational RAG Agent

**Stack: Python, LangChain, FAISS, Hugging Face, OpenAI API**

A retrieval-augmented conversational agent with:

- **RAG pipeline** — documents chunked, embedded (OpenAI), indexed in FAISS
- **Retrieval optimisation** — FAISS retrieves a wide candidate set, a Hugging Face cross-encoder reranks for true relevance
- **Prompt engineering** — grounded generation prompt that refuses to answer outside retrieved context
- **Conversation memory** — multi-turn chat via LangChain, follow-up questions resolve correctly
- **Evaluation framework** — labeled test set, baseline vs RAG accuracy, measured not assumed
- **Monitoring framework** — per-query latency, retrieval scores, and response quality logged

In [ ]:
import os
import time
import pandas as pd
from dotenv import load_dotenv

# LangChain: loaders, splitting, chains, memory
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# Hugging Face: cross-encoder reranker
from sentence_transformers import CrossEncoder

load_dotenv()
print("all good")

## 1. Load and chunk documents (LangChain)

`DirectoryLoader` reads every `.txt` in `docs/`. `RecursiveCharacterTextSplitter` chunks them — 500 characters with 100 overlap, splitting on natural boundaries (paragraphs → sentences → words) before falling back to hard cuts.

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

In [ ]:
import os
print(os.listdir(r"C:\Users\reshm\rag-agent"))

In [ ]:
loader = DirectoryLoader(r"C:\Users\reshm\rag-agent\docs", glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
documents = loader.load()

# strip metadata headers so they don't pollute retrieval
for doc in documents:
    if "\n---\n" in doc.page_content:
        doc.page_content = doc.page_content.split("\n---\n", 1)[1]

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"Loaded {len(documents)} documents → {len(chunks)} chunks")

## 2. Embed and index (OpenAI API + FAISS)

Each chunk is embedded with OpenAI's `text-embedding-3-small` and stored in a FAISS index for fast similarity search.

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"FAISS index holds {vectorstore.index.ntotal} vectors")

## 3. Retrieval optimisation: Hugging Face cross-encoder reranking

Two-stage retrieval:

1. **FAISS (fast, wide net)** — embedding similarity pulls the top 10 candidate chunks. Fast, but vector similarity is a rough proxy for relevance.
2. **Cross-encoder (accurate, narrow)** — a Hugging Face model (`cross-encoder/ms-marco-MiniLM-L-6-v2`) reads each (question, chunk) pair *together* and scores actual relevance. The top 3 after reranking go to the LLM.

Why it works: the embedding model encodes question and chunks separately, so it can only compare them geometrically. The cross-encoder attends across both at once, catching relevance that embedding distance misses. This is the "retrieval optimisation" step — measurably better context selection than raw FAISS top-k.

In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve(question, fetch_k=10, top_k=3):
    """Two-stage retrieval: FAISS wide net → cross-encoder rerank."""
    candidates = vectorstore.similarity_search_with_score(question, k=fetch_k)
    docs = [d for d, _ in candidates]
    faiss_scores = [float(s) for _, s in candidates]

    pairs = [(question, d.page_content) for d in docs]
    rerank_scores = reranker.predict(pairs)

    ranked = sorted(zip(docs, faiss_scores, rerank_scores), key=lambda x: x[2], reverse=True)
    top = ranked[:top_k]

    return {
        "docs": [d for d, _, _ in top],
        "faiss_scores": [f for _, f, _ in top],
        "rerank_scores": [float(r) for _, _, r in top],
    }

# Quick check
r = retrieve("What is the difference between a Docker image and a container?")
for i, (doc, rs) in enumerate(zip(r["docs"], r["rerank_scores"])):
    print(f"--- chunk {i} (rerank score {rs:.2f}) ---")
    print(doc.page_content[:150], "\n")

## 4. Grounded generation (OpenAI API + prompt engineering)

The prompt explicitly restricts the model to the retrieved context and instructs it to say "I don't know" when the answer isn't there — this is what keeps the system grounded instead of hallucinating.

In [ ]:
qa_prompt = PromptTemplate(
    template="""Answer using only the provided context. If the context does not contain the answer, say you don't know.

Context:
{context}

Question: {question}"""
    input_variables=["context", "question"],
)

llm = ChatOpenAI(model="gpt-4o-mini")

def generate(question, retrieved_docs):
    context = "\n---\n".join(d.page_content for d in retrieved_docs)
    return llm.invoke(qa_prompt.format(context=context, question=question)).content

## 5. Conversational memory (LangChain)

`ConversationBufferMemory` keeps the chat history. Before retrieving, the current question is rewritten ("condensed") using that history, so follow-ups like "how is *it* different from a container?" resolve correctly — the standalone question sent to retrieval becomes "How is a Docker image different from a container?"

In [ ]:
class SimpleMemory:
    def __init__(self):
        self.history = []

    def load_memory_variables(self, _):
        return {"chat_history": self.history}

    def save_context(self, inputs, outputs):
        self.history.append({"input": inputs.get("input"), "output": outputs.get("output")})

memory = SimpleMemory()

condense_prompt = PromptTemplate(
    template="""Given the conversation history and a follow-up question, rewrite the follow-up as a standalone question. If it is already standalone, return it unchanged.

History:
{chat_history}

Follow-up question: {question}

Standalone question:"""
    input_variables=["chat_history", "question"],
)

def condense(question):
    history = memory.load_memory_variables({})["chat_history"]
    if not history:
        return question
    history_text = "\n".join(f"{m['input']}: {m['output']}" for m in history)
    return llm.invoke(condense_prompt.format(chat_history=history_text, question=question)).content.strip()

def chat(question):
    standalone = condense(question)
    retrieved = retrieve(standalone)
    answer = generate(standalone, retrieved["docs"])
    memory.save_context({"input": question}, {"output": answer})
    return answer

# Multi-turn test — turn 2 only works because of memory
print("Turn 1:", chat("What is a Docker image?"))
print()
print("Turn 2:", chat("How is it different from a container?"))

## 6. Evaluation framework

Labeled test set → run each question through a **baseline** (LLM alone, no retrieval) and the **full RAG pipeline** → grade both answers against the expected answer with a temperature-0 LLM grader → compare accuracy.

The improvement number this produces is measured, not assumed. Expand `eval_set` to 15–20 questions drawn from your `docs/` content before trusting the percentage.

In [ ]:
eval_set = [
    {"question": "What three conversation-layer products did Twilio announce at Signal 2026?",
     "expected": "Conversation Orchestrator, Conversation Memory, and Conversation Intelligence."},
    {"question": "What does Conversation Memory do according to the announcement?",
     "expected": "It provides shared customer memory, preserving context so interactions are seamless and customers don't have to repeat themselves."},
    {"question": "What is Agent Connect?",
     "expected": "An open-source tool for plugging third-party AI agents into Twilio's communication infrastructure, using any model or framework."},
    {"question": "What was the theme of Signal 2026?",
     "expected": "Build Wonder."},
    {"question": "What does Conversation Intelligence do?",
     "expected": "It provides real-time intent, sentiment, and risk monitoring during live customer conversations."},
    {"question": "What does Conversation Orchestrator do?",
     "expected": "It coordinates and routes interactions, turning fragmented interactions into a single continuous conversation across channels."},
    {"question": "Where and when did Signal 2026 take place?",
     "expected": "May 6-7, 2026, at the Marriott Marquis in San Francisco."},
    {"question": "What console update did Twilio announce at Signal 2026?",
     "expected": "A complete redesign of the Twilio Console, a unified interface with an integrated AI assistant."},
]

grader = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def grade(question, expected, actual):
    p = f"""You are grading an answer for correctness.\n\nQuestion: {question}\nExpected answer: {expected}\nActual answer: {actual}\n\nDoes the actual answer correctly convey the same meaning as the expected answer? Reply with exactly one word: CORRECT or INCORRECT."""
    return grader.invoke(p).content.strip().upper().startswith("CORRECT")

def answer_baseline(question):
    return llm.invoke(question).content

def answer_rag(question):
    return generate(question, retrieve(question)["docs"])

rows = []
for item in eval_set:
    q, exp = item["question"], item["expected"]
    b_ans, r_ans = answer_baseline(q), answer_rag(q)
    rows.append({
        "question": q,
        "baseline_correct": grade(q, exp, b_ans),
        "rag_correct": grade(q, exp, r_ans),
    })

eval_df = pd.DataFrame(rows)
b_acc = eval_df["baseline_correct"].mean() * 100
r_acc = eval_df["rag_correct"].mean() * 100
print(f"Baseline accuracy: {b_acc:.1f}%")
print(f"RAG accuracy:      {r_acc:.1f}%")
print(f"Improvement:       {r_acc - b_acc:+.1f} percentage points")
eval_df

In [ ]:
r = retrieve("What three conversation-layer products did Twilio announce at Signal 2026?")
for i, d in enumerate(r["docs"]):
    print(f"--- chunk {i} ---")
    print(d.page_content[:300], "\n")

In [ ]:
for doc in documents:
    if "\n---\n" in doc.page_content:
        doc.page_content = doc.page_content.split("\n---\n", 1)[1]

In [ ]:
def answer_rag(question):
    return generate(question, retrieve(question, fetch_k=10, top_k=5)["docs"])

## 7. Monitoring framework

Every query through `monitored_chat` logs:

- **Latency** — total wall-clock seconds for retrieve + rerank + generate
- **Retrieval accuracy signal** — top FAISS distance and top rerank score for the chunks used
- **Response quality** — graded against an expected answer when one is provided, else left for manual review

In a production system this log would go to a proper observability stack; here it's an inspectable dataframe.

In [ ]:
monitoring_log = []

def monitored_chat(question, expected=None):
    start = time.time()
    standalone = condense(question)
    retrieved = retrieve(standalone)
    answer = generate(standalone, retrieved["docs"])
    memory.save_context({"input": question}, {"output": answer})
    latency = time.time() - start

    monitoring_log.append({
        "question": question,
        "standalone_question": standalone,
        "latency_sec": round(latency, 2),
        "top_faiss_distance": round(retrieved["faiss_scores"][0], 4),
        "top_rerank_score": round(retrieved["rerank_scores"][0], 4),
        "quality_correct": grade(question, expected, answer) if expected else None,
        "answer": answer,
    })
    return answer

print(monitored_chat("What objects does Docker manage?"))
print()
print(monitored_chat("Who won the 2022 World Cup?"))  # not in docs — should say I don't know

monitoring_df = pd.DataFrame(monitoring_log)
monitoring_df

## Summary — what each tool does

| Tool | Role |
|---|---|
| **LangChain** | Document loading, chunking, memory, prompt templates |
| **OpenAI API** | Embeddings (`text-embedding-3-small`) + generation (`gpt-4o-mini`) |
| **FAISS** | Fast first-stage vector similarity search (wide candidate net) |
| **Hugging Face** | Cross-encoder reranker — second-stage retrieval optimisation |
| **Python** | Everything else: eval harness, monitoring log, glue |